# Notebook 12 — Front Behavior Labeling

## Mục tiêu
Tạo **ground truth hành vi thủ công** cho các cửa sổ 5 giây từ Notebook 11.

### Phân biệt bắt buộc
- `context` = điều kiện thí nghiệm/video đã biết.
- `behavior_label` = hành vi **quan sát được** trong cửa sổ và được người nghiên cứu gán thủ công.
- Không tự động biến `context` thành `behavior_label`.
- Không dùng nhãn “stress” nếu chưa có ground truth sinh học.

### Nhãn khởi tạo
- `NORMAL_SWIM`
- `LOW_ACTIVITY`
- `FEEDING`
- `SHELTER_TRANSITION`
- `PAIR_INTERACTION`
- `UNCERTAIN`
- `EXCLUDE`

## Bản sửa phát hình — GIF tự chạy
Trong VS Code Notebook, `ipywidgets.Play` có thể chỉ nhảy từng frame và không tự chạy ổn định.
Bản này **bỏ nút Play** và thay bằng:

- **GIF động tự chạy/lặp** cho toàn bộ cửa sổ 5 giây;
- thanh trượt frame vẫn giữ để soi từng frame thủ công;
- không cần FFmpeg/MP4 codec.

GIF chỉ phục vụ review. Dữ liệu tracking, feature và nhãn lưu không thay đổi.

In [1]:
# ============================================================
# 0. SETUP
# ============================================================
from pathlib import Path
import json, math, sys
import numpy as np
import pandas as pd
import cv2

from IPython.display import display, clear_output

def find_project_root():
    candidates = [Path("/home/diy-hus/fish"), Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p.resolve()
    return Path("/home/diy-hus/fish").resolve()

PROJECT_ROOT = find_project_root()
FEATURE_PATH = PROJECT_ROOT / "results/behavior/front_individual_behavior_features.csv"
TRAJ_PATH = PROJECT_ROOT / "results/trajectory/front_cleaned_trajectories.csv"
RESULTS_DIR = PROJECT_ROOT / "results/behavior"
CLIP_DIR = PROJECT_ROOT / "outputs/front/behavior_labeling/clips"
LOG_DIR = PROJECT_ROOT / "logs/behavior/FRONT_BEHAVIOR_LABELING_001"
for p in [RESULTS_DIR, CLIP_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_PATH = RESULTS_DIR / "front_behavior_labels.csv"
MANIFEST_PATH = RESULTS_DIR / "front_behavior_labeling_manifest.csv"

assert FEATURE_PATH.exists(), "Run Notebook 11 first."
assert TRAJ_PATH.exists(), "Run Notebook 10 first."
from io import BytesIO
from PIL import Image as PILImage


In [2]:
# ============================================================
# 1. CONFIG
# ============================================================
EXPERIMENT_ID = "FRONT_BEHAVIOR_LABELING_001"

VIDEO_FILES = {
    "V4_BREEDING": "4.mp4",
    "V8_PAIR_T1": "8.mp4",
    "V5_NORMAL": "5.mp4",
    "V3_FEEDING": "3.mp4",
}

ANNOTATION_TAXONOMY = [
    "NORMAL_SWIM",
    "LOW_ACTIVITY",
    "FEEDING",
    "SHELTER_TRANSITION",
    "PAIR_INTERACTION",
    "UNCERTAIN",
    "EXCLUDE",
]

# Final class set selected AFTER annotation audit.
# LOW_ACTIVITY is not modeled because no confirmed GT windows were observed.
# UNCERTAIN / EXCLUDE are QC states, not behavior classes.
MODEL_BEHAVIOR_CLASSES = [
    "NORMAL_SWIM",
    "PAIR_INTERACTION",
    "SHELTER_TRANSITION",
    "FEEDING",
]

# Backward-compatible name used by the labeling GUI.
PRIMARY_LABELS = ANNOTATION_TAXONOMY
LABELING_TARGET_PER_CONTEXT = 40
RANDOM_SEED = 42

# Để giảm trùng lặp mạnh giữa các cửa sổ 5s/step1s:
# chỉ lấy candidate bắt đầu gần bội số 5 giây.
CANDIDATE_STRIDE_SEC = 5.0

CLIP_FPS_MODE = "SOURCE"
DRAW_TRAJECTORY_TAIL_SEC = 1.0

# GUI review playback rate. Lower than source FPS to keep VS Code widgets responsive.
REVIEW_PLAYBACK_FPS = 6.0

print("Annotation taxonomy:", ANNOTATION_TAXONOMY)
print("Model behavior classes:", MODEL_BEHAVIOR_CLASSES)
print("Target/context:", LABELING_TARGET_PER_CONTEXT)

# GIF review only; does not alter research data.
REVIEW_GIF_WIDTH = 640
REVIEW_GIF_FPS = 6.0


Annotation taxonomy: ['NORMAL_SWIM', 'LOW_ACTIVITY', 'FEEDING', 'SHELTER_TRANSITION', 'PAIR_INTERACTION', 'UNCERTAIN', 'EXCLUDE']
Model behavior classes: ['NORMAL_SWIM', 'PAIR_INTERACTION', 'SHELTER_TRANSITION', 'FEEDING']
Target/context: 40


In [3]:
# ============================================================
# 2. BUILD / PRESERVE LABELING MANIFEST
# ============================================================
feat = pd.read_csv(FEATURE_PATH)
traj = pd.read_csv(TRAJ_PATH)

# Candidate non-overlapping windows.
candidate = feat[
    np.isclose(
        np.mod(feat["window_start_sec"], CANDIDATE_STRIDE_SEC),
        0.0,
        atol=1e-6,
    )
].copy()

sampled_parts = []
rng = np.random.default_rng(RANDOM_SEED)

for context, sub in candidate.groupby("context"):
    n = min(LABELING_TARGET_PER_CONTEXT, len(sub))
    if n == 0:
        continue
    chosen_idx = rng.choice(sub.index.to_numpy(), size=n, replace=False)
    sampled_parts.append(sub.loc[chosen_idx])

manifest = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else pd.DataFrame()

manifest = manifest[[
    "window_id","video_id","context","pred_track_id","trajectory_uid",
    "window_start_sec","window_end_sec","coverage_ratio","observed_ratio"
]].copy()
manifest["label_status"] = "UNLABELED"

# Preserve existing label status/labels.
if LABEL_PATH.exists():
    old_labels = pd.read_csv(LABEL_PATH)
else:
    old_labels = pd.DataFrame(columns=[
        "window_id","behavior_label","label_certainty","annotation_note"
    ])

if len(old_labels):
    manifest = manifest.merge(
        old_labels[["window_id","behavior_label","label_certainty","annotation_note"]],
        on="window_id",
        how="left",
    )
    manifest.loc[manifest["behavior_label"].notna(), "label_status"] = "LABELED"
else:
    manifest["behavior_label"] = pd.NA
    manifest["label_certainty"] = pd.NA
    manifest["annotation_note"] = pd.NA

manifest = manifest.sort_values(["context","video_id","window_start_sec","pred_track_id"]).reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)

print("Labeling manifest rows:", len(manifest))
display(manifest.groupby("context").agg(
    candidates=("window_id","count"),
    labeled=("label_status", lambda s: int(s.eq("LABELED").sum()))
))

Labeling manifest rows: 98


,candidates,labeled
context,,
BREEDING_PARENTAL_CARE,13,13
FEEDING,40,40
NORMAL,40,40
PAIR_T1,5,5


In [ ]:
# ============================================================
# 3. REVIEW FRAME GENERATOR — JPEG SLIDER + ANIMATED GIF
# ============================================================

REVIEW_JPEG_QUALITY = 82


def _video_meta(path):
    cap = cv2.VideoCapture(str(path))

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")

    meta = {
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }

    cap.release()

    if (
        meta["fps"] <= 0
        or meta["width"] <= 0
        or meta["height"] <= 0
    ):
        raise RuntimeError(
            f"Invalid video metadata: {path} -> {meta}"
        )

    return meta


def _resize_for_review(
    frame,
    max_width=REVIEW_GIF_WIDTH,
):
    h, w = frame.shape[:2]

    if w <= max_width:
        return frame

    scale = max_width / float(w)

    return cv2.resize(
        frame,
        (
            int(round(w * scale)),
            int(round(h * scale)),
        ),
        interpolation=cv2.INTER_AREA,
    )


def _encode_jpeg(frame):
    ok, buf = cv2.imencode(
        ".jpg",
        frame,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            int(REVIEW_JPEG_QUALITY),
        ],
    )

    if not ok:
        raise RuntimeError("JPEG_ENCODE_FAILED")

    return buf.tobytes()


def _frames_to_gif(frames_jpeg, review_fps):
    """
    Build an animated GIF entirely in memory.
    Reliable in VS Code/Jupyter because it is rendered as an image,
    not as a video/widget timer.
    """

    if not frames_jpeg:
        raise RuntimeError("NO_FRAMES_FOR_GIF")

    pil_frames = []

    for b in frames_jpeg:
        img = PILImage.open(BytesIO(b)).convert("RGB")
        # Adaptive palette keeps GIF reasonably small.
        img = img.convert(
            "P",
            palette=PILImage.Palette.ADAPTIVE,
            colors=128,
        )
        pil_frames.append(img)

    duration_ms = max(
        80,
        int(round(1000.0 / float(review_fps))),
    )

    out = BytesIO()

    pil_frames[0].save(
        out,
        format="GIF",
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
        disposal=2,
    )

    return out.getvalue()


def build_review_frames(row):
    """
    Read the complete 5 s source interval, sample it to ~REVIEW_GIF_FPS
    for display, and return both:
    - JPEG frames for manual slider review
    - animated GIF bytes for automatic playback

    This affects GUI display only.
    """

    video_id = str(row["video_id"])
    uid = str(row["trajectory_uid"])
    start = float(row["window_start_sec"])
    end = float(row["window_end_sec"])

    video_path = (
        PROJECT_ROOT
        / "data/raw/front"
        / VIDEO_FILES[video_id]
    )

    if not video_path.exists():
        raise FileNotFoundError(video_path)

    meta = _video_meta(video_path)
    source_fps = float(meta["fps"])

    display_stride = max(
        1,
        int(
            round(
                source_fps
                / float(REVIEW_GIF_FPS)
            )
        ),
    )

    review_fps = source_fps / display_stride

    start_frame = int(round(start * source_fps))
    end_frame_exclusive = int(round(end * source_fps))

    target = traj[
        traj["trajectory_uid"].astype(str).eq(uid)
        & traj["time_sec"].ge(start)
        & traj["time_sec"].lt(end)
    ].copy()

    target_by_frame = {
        int(r["frame_index"]): r
        for _, r in target.iterrows()
    }

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open raw video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    tail_points = []

    tail_frames = max(
        1,
        int(
            round(
                DRAW_TRAJECTORY_TAIL_SEC
                * source_fps
            )
        ),
    )

    frames_jpeg = []
    source_frame_indices = []

    for fi in range(
        start_frame,
        end_frame_exclusive,
    ):
        ok, frame = cap.read()

        if not ok:
            break

        r = target_by_frame.get(fi)

        if r is not None:
            cx = int(round(float(r["cx_clean"])))
            cy = int(round(float(r["cy_clean"])))

            bw = float(r["bbox_w"])
            bh = float(r["bbox_h"])

            x1 = int(round(cx - bw / 2))
            y1 = int(round(cy - bh / 2))
            x2 = int(round(cx + bw / 2))
            y2 = int(round(cy + bh / 2))

            x1 = max(0, min(x1, meta["width"] - 1))
            y1 = max(0, min(y1, meta["height"] - 1))
            x2 = max(x1 + 1, min(x2, meta["width"]))
            y2 = max(y1 + 1, min(y2, meta["height"]))

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2,
            )

            cv2.putText(
                frame,
                f"TARGET Track ID {int(r['pred_track_id'])}",
                (
                    max(0, x1),
                    max(25, y1 - 8),
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

            tail_points.append(
                (
                    fi,
                    (cx, cy),
                )
            )

        tail_points = [
            (f, p)
            for f, p in tail_points
            if fi - f <= tail_frames
        ]

        pts = [p for _, p in tail_points]

        for a, b in zip(
            pts[:-1],
            pts[1:],
        ):
            cv2.line(
                frame,
                a,
                b,
                (255, 255, 255),
                2,
            )

        elapsed = (
            (fi - start_frame)
            / source_fps
        )

        cv2.putText(
            frame,
            (
                f"{start:.1f}-{end:.1f}s "
                f"| t={elapsed:.2f}s "
                f"| context={row['context']}"
            ),
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        should_display = (
            ((fi - start_frame) % display_stride == 0)
            or fi == end_frame_exclusive - 1
        )

        if should_display:
            frame_small = _resize_for_review(frame)

            frames_jpeg.append(
                _encode_jpeg(frame_small)
            )

            source_frame_indices.append(fi)

    cap.release()

    if not frames_jpeg:
        raise RuntimeError("NO_REVIEW_FRAMES_GENERATED")

    gif_bytes = _frames_to_gif(
        frames_jpeg,
        review_fps,
    )

    return {
        "frames_jpeg": frames_jpeg,
        "gif_bytes": gif_bytes,
        "source_frame_indices": source_frame_indices,
        "fps": source_fps,
        "review_fps": review_fps,
        "display_stride": display_stride,
        "start_sec": start,
        "end_sec": end,
    }


print("Animated GIF review generator ready.")
print("No ipywidgets.Play / MP4 / ffmpeg required.")

In [ ]:
# ============================================================
# 4. SIMPLE LABELING GUI — AUTO-LOOP GIF + MANUAL FRAME SLIDER
# ============================================================

import ipywidgets as widgets


state = {
    "row_index": None,
    "review": None,
}


label_dd = widgets.Dropdown(
    options=PRIMARY_LABELS,
    description="Behavior:",
)

certainty_dd = widgets.Dropdown(
    options=[
        "CERTAIN",
        "UNCERTAIN",
    ],
    value="CERTAIN",
    description="Certainty:",
)

note_box = widgets.Text(
    description="Note:",
)

next_btn = widgets.Button(
    description="Load next unlabeled",
    button_style="info",
)

save_btn = widgets.Button(
    description="Save label + next",
    button_style="success",
)

reload_btn = widgets.Button(
    description="Reload preview",
)

status = widgets.HTML()

gif_widget = widgets.Image(
    format="gif",
)

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Frame:",
    continuous_update=True,
)

manual_image = widgets.Image(
    format="jpeg",
)

frame_info = widgets.HTML()


def _load_labels():
    if LABEL_PATH.exists():
        return pd.read_csv(LABEL_PATH)

    return pd.DataFrame(
        columns=[
            "window_id",
            "video_id",
            "context",
            "pred_track_id",
            "trajectory_uid",
            "window_start_sec",
            "window_end_sec",
            "behavior_label",
            "label_certainty",
            "annotation_note",
        ]
    )


def _save_label(
    row,
    behavior_label,
    certainty,
    note,
):
    labels = _load_labels()

    labels = labels[
        ~labels["window_id"]
        .astype(str)
        .eq(
            str(
                row["window_id"]
            )
        )
    ].copy()

    new = pd.DataFrame(
        [
            {
                "window_id": row["window_id"],
                "video_id": row["video_id"],
                "context": row["context"],
                "pred_track_id": int(row["pred_track_id"]),
                "trajectory_uid": row["trajectory_uid"],
                "window_start_sec": float(row["window_start_sec"]),
                "window_end_sec": float(row["window_end_sec"]),
                "behavior_label": behavior_label,
                "label_certainty": certainty,
                "annotation_note": note,
            }
        ]
    )

    labels = pd.concat(
        [labels, new],
        ignore_index=True,
    )

    labels.to_csv(
        LABEL_PATH,
        index=False,
    )


def _next_unlabeled_index():
    labels = _load_labels()

    done = (
        set(
            labels["window_id"].astype(str)
        )
        if len(labels)
        else set()
    )

    for i, row in manifest.iterrows():
        if str(row["window_id"]) not in done:
            return i

    return None


def _show_manual_frame(index):
    review = state["review"]

    if review is None:
        return

    index = int(index)

    if not (
        0
        <= index
        < len(review["frames_jpeg"])
    ):
        return

    manual_image.value = review["frames_jpeg"][index]

    source_fi = review["source_frame_indices"][index]

    elapsed = (
        (
            source_fi
            - review["source_frame_indices"][0]
        )
        / review["fps"]
    )

    frame_info.value = (
        f"<b>Manual frame:</b> "
        f"{index+1}/{len(review['frames_jpeg'])}"
        f" | source frame={source_fi}"
        f" | t={elapsed:.2f}s"
    )


def _slider_changed(change):
    if change["name"] == "value":
        _show_manual_frame(
            change["new"]
        )


slider.observe(
    _slider_changed,
    names="value",
)


def _render_current():
    i = state["row_index"]

    if i is None:
        print("Load a window first.")
        return

    row = manifest.loc[i]

    review = build_review_frames(row)
    state["review"] = review

    n = len(review["frames_jpeg"])

    slider.min = 0
    slider.max = max(0, n - 1)
    slider.value = 0

    gif_widget.value = review["gif_bytes"]

    clear_output(wait=True)

    display(
        widgets.HBox(
            [
                label_dd,
                certainty_dd,
                note_box,
                next_btn,
                save_btn,
                reload_btn,
            ]
        )
    )

    labeled = len(_load_labels())

    status.value = (
        f"<b>{row['video_id']} "
        f"| context={row['context']} "
        f"| Track ID={int(row['pred_track_id'])}</b>"
        f"<br>window="
        f"{row['window_start_sec']:.1f}-"
        f"{row['window_end_sec']:.1f}s"
        f"<br>labeled={labeled}/{len(manifest)}"
        f"<br>review frames={n}"
        f" | source FPS={review['fps']:.3f}"
        f" | GIF FPS={review['review_fps']:.2f}"
        f" | stride={review['display_stride']}"
    )

    display(status)

    display(
        widgets.HTML(
            "<b>Animated 5-second preview (auto-loop):</b>"
        )
    )
    display(gif_widget)

    display(
        widgets.HTML(
            "<b>Manual frame inspection:</b>"
        )
    )
    display(slider)
    display(frame_info)
    display(manual_image)

    _show_manual_frame(0)


def _show_row(i):
    state["row_index"] = int(i)
    _render_current()


def _on_next(_):
    i = _next_unlabeled_index()

    if i is None:
        clear_output(wait=True)
        print("LABELING_MANIFEST_COMPLETE")
        return

    _show_row(i)


def _on_save(_):
    i = state["row_index"]

    if i is None:
        print("Load a window first.")
        return

    row = manifest.loc[i]

    _save_label(
        row,
        label_dd.value,
        certainty_dd.value,
        note_box.value,
    )

    note_box.value = ""

    _on_next(None)


def _on_reload(_):
    if state["row_index"] is None:
        print("Load a window first.")
        return

    _render_current()


next_btn.on_click(_on_next)
save_btn.on_click(_on_save)
reload_btn.on_click(_on_reload)


display(
    widgets.HBox(
        [
            label_dd,
            certainty_dd,
            note_box,
            next_btn,
            save_btn,
            reload_btn,
        ]
    )
)

display(status)

print("Click 'Load next unlabeled'.")

In [ ]:
# ============================================================
# 4A. OPTIONAL GIF PREVIEW DIAGNOSTIC
# ============================================================

i = _next_unlabeled_index()

if i is None:
    print("No unlabeled window remains.")
else:
    row = manifest.loc[i]
    review = build_review_frames(row)

    print("review frames:", len(review["frames_jpeg"]))
    print("source fps:", review["fps"])
    print("gif fps:", review["review_fps"])
    print("display stride:", review["display_stride"])
    print("gif bytes:", len(review["gif_bytes"]))

    test_gif = widgets.Image(
        value=review["gif_bytes"],
        format="gif",
    )

    display(test_gif)

In [4]:
# ============================================================
# 5. LABEL AUDIT — FINAL 4-CLASS MODEL SET
# ============================================================

labels = (
    pd.read_csv(LABEL_PATH)
    if LABEL_PATH.exists()
    else pd.DataFrame()
)

if len(labels):
    audit = (
        labels.groupby(
            ["behavior_label", "label_certainty"]
        )
        .size()
        .rename("windows")
        .reset_index()
        .sort_values(
            ["behavior_label", "label_certainty"]
        )
    )

    display(audit)

    certain_all = labels[
        labels["label_certainty"]
        .astype(str)
        .str.upper()
        .eq("CERTAIN")
    ].copy()

    model_labels = certain_all[
        certain_all["behavior_label"]
        .isin(MODEL_BEHAVIOR_CLASSES)
    ].copy()

    class_counts = (
        model_labels["behavior_label"]
        .value_counts()
        .reindex(
            MODEL_BEHAVIOR_CLASSES,
            fill_value=0,
        )
    )

    print(
        "Labeled:",
        len(labels),
        "/",
        len(manifest),
    )

    print(
        "CERTAIN model-trainable:",
        len(model_labels),
    )

    print(
        "4-class counts:",
        class_counts.to_dict(),
    )

    print(
        "Model classes:",
        MODEL_BEHAVIOR_CLASSES,
    )

    omitted = sorted(
        set(
            certain_all["behavior_label"]
            .dropna()
            .astype(str)
        )
        - set(MODEL_BEHAVIOR_CLASSES)
    )

    print(
        "CERTAIN labels omitted from model:",
        omitted,
    )

    if int(class_counts.get("FEEDING", 0)) < 15:
        print(
            "WARNING: FEEDING has fewer than 15 CERTAIN windows; "
            "interpret per-class CV metrics cautiously."
        )
else:
    print("No labels yet.")

,behavior_label,label_certainty,windows
0,FEEDING,CERTAIN,7
1,NORMAL_SWIM,CERTAIN,54
2,PAIR_INTERACTION,CERTAIN,21
3,SHELTER_TRANSITION,CERTAIN,16


Labeled: 98 / 98
CERTAIN model-trainable: 98
4-class counts: {'NORMAL_SWIM': 54, 'PAIR_INTERACTION': 21, 'SHELTER_TRANSITION': 16, 'FEEDING': 7}
Model classes: ['NORMAL_SWIM', 'PAIR_INTERACTION', 'SHELTER_TRANSITION', 'FEEDING']
CERTAIN labels omitted from model: []


In [5]:
# ============================================================
# 6. FINAL SUMMARY — ANNOTATION + FINAL MODEL CLASS SET
# ============================================================

labels = (
    pd.read_csv(LABEL_PATH)
    if LABEL_PATH.exists()
    else pd.DataFrame()
)

if len(labels):
    certain_model = labels[
        labels["label_certainty"]
        .astype(str)
        .str.upper()
        .eq("CERTAIN")
        &
        labels["behavior_label"]
        .isin(MODEL_BEHAVIOR_CLASSES)
    ].copy()

    model_class_counts = (
        certain_model["behavior_label"]
        .value_counts()
        .reindex(
            MODEL_BEHAVIOR_CLASSES,
            fill_value=0,
        )
        .astype(int)
        .to_dict()
    )
else:
    certain_model = pd.DataFrame()
    model_class_counts = {
        c: 0
        for c in MODEL_BEHAVIOR_CLASSES
    }

summary = {
    "experiment_id": EXPERIMENT_ID,
    "manifest_windows": int(len(manifest)),
    "labeled_windows": int(len(labels)),
    "annotation_taxonomy": ANNOTATION_TAXONOMY,
    "model_behavior_classes": MODEL_BEHAVIOR_CLASSES,
    "model_class_count": int(len(MODEL_BEHAVIOR_CLASSES)),
    "model_trainable_windows": int(len(certain_model)),
    "model_class_counts": model_class_counts,
    "taxonomy_decision_note": (
        "LOW_ACTIVITY excluded from modeling because no confirmed GT windows "
        "were observed. UNCERTAIN and EXCLUDE remain annotation QC states."
    ),
    "outputs": [
        str(
            MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        str(
            LABEL_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    ],
}

(
    LOG_DIR
    / "summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print("FINAL SUMMARY")
print(
    json.dumps(
        summary,
        indent=2,
    )
)

FINAL SUMMARY
{
  "experiment_id": "FRONT_BEHAVIOR_LABELING_001",
  "manifest_windows": 98,
  "labeled_windows": 98,
  "annotation_taxonomy": [
    "NORMAL_SWIM",
    "LOW_ACTIVITY",
    "FEEDING",
    "SHELTER_TRANSITION",
    "PAIR_INTERACTION",
    "UNCERTAIN",
    "EXCLUDE"
  ],
  "model_behavior_classes": [
    "NORMAL_SWIM",
    "PAIR_INTERACTION",
    "SHELTER_TRANSITION",
    "FEEDING"
  ],
  "model_class_count": 4,
  "model_trainable_windows": 98,
  "model_class_counts": {
    "NORMAL_SWIM": 54,
    "PAIR_INTERACTION": 21,
    "SHELTER_TRANSITION": 16,
    "FEEDING": 7
  },
  "taxonomy_decision_note": "LOW_ACTIVITY excluded from modeling because no confirmed GT windows were observed. UNCERTAIN and EXCLUDE remain annotation QC states.",
  "outputs": [
    "results/behavior/front_behavior_labeling_manifest.csv",
    "results/behavior/front_behavior_labels.csv"
  ]
}
